# The Least Round Way — código y experimentos
Implementación top-down con memoización, pruebas de correctitud y benchmark hasta n=1000.

In [ ]:
from functools import lru_cache
import sys, random, time, statistics, gc, platform
import matplotlib.pyplot as plt

# Para n=1000 la profundidad de una ruta recursiva puede acercarse a 2n-1.
sys.setrecursionlimit(1_000_000)

def exponent(x, p):
    if x == 0:
        return None
    c = 0
    while x % p == 0:
        c += 1
        x //= p
    return c

def solve_factor(a, p, reconstruct=True):
    n = len(a)
    INF = 10**9

    cost = [
        [INF if a[i][j] == 0 else exponent(a[i][j], p) for j in range(n)]
        for i in range(n)
    ]

    choice = [[None] * n for _ in range(n)] if reconstruct else None

    @lru_cache(None)
    def D(i, j):
        if i >= n or j >= n:
            return INF

        if i == n - 1 and j == n - 1:
            return cost[i][j]

        down = D(i + 1, j)
        right = D(i, j + 1)

        if reconstruct:
            choice[i][j] = "D" if down <= right else "R"

        return cost[i][j] + min(down, right)

    value = D(0, 0)

    if not reconstruct:
        return value, ""

    path = []
    i = j = 0
    while i != n - 1 or j != n - 1:
        move = choice[i][j]
        path.append(move)
        if move == "D":
            i += 1
        else:
            j += 1

    return value, "".join(path)

def path_through_zero(n, zi, zj):
    return "D" * zi + "R" * zj + "D" * (n - 1 - zi) + "R" * (n - 1 - zj)

def least_round_way(a):
    n = len(a)
    zero = None

    for i in range(n):
        for j in range(n):
            if a[i][j] == 0 and zero is None:
                zero = (i, j)

    v2, p2 = solve_factor(a, 2)
    v5, p5 = solve_factor(a, 5)

    best, path = (v2, p2) if v2 <= v5 else (v5, p5)

    if zero is not None and best > 1:
        zi, zj = zero
        return 1, path_through_zero(n, zi, zj)

    return best, path


In [ ]:
tests = [
    [[1,2,3],[4,5,6],[7,8,9]],
    [[10,10],[10,10]],
    [[1,10,1],[1,0,1],[10,10,10]],
    [[1,1],[1,1]],
]

for i, matrix in enumerate(tests, 1):
    print(f"Caso {i}: {least_round_way(matrix)}")


In [ ]:
def benchmark_instance(a):
    v2, _ = solve_factor(a, 2, reconstruct=False)
    v5, _ = solve_factor(a, 5, reconstruct=False)
    return min(v2, v5)

print("Python:", sys.version.split()[0])
print("Plataforma:", platform.platform())
print("Límite de recursión:", sys.getrecursionlimit())

rng = random.Random(20261005)
sizes = [50, 100, 150, 200, 300, 400, 500, 600, 800, 1000]
times = []

for n in sizes:
    a = [[rng.randint(1, 10**9) for _ in range(n)] for _ in range(n)]
    samples = []
    for _ in range(3):
        gc.collect()
        t0 = time.perf_counter()
        benchmark_instance(a)
        samples.append(time.perf_counter() - t0)
    times.append(statistics.median(samples))

for n, t in zip(sizes, times):
    print(f"n={n}: {t:.6f} s")

plt.plot(sizes, times, marker="o")
plt.xlabel("Tamaño de la matriz n")
plt.ylabel("Tiempo mediano (s)")
plt.title("Tiempo de ejecución vs. tamaño de entrada")
plt.grid(True, alpha=0.25)
plt.show()
